# Multimodal Embedding with Qwen3-VL and OpenVINO

The [Qwen3-VL-Embedding model series](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) is built upon the powerful Qwen3-VL foundation model, specifically designed for multimodal embedding tasks. It accepts diverse inputs including text, images, screenshots, and videos, as well as inputs containing a mixture of these modalities. The model generates high-dimensional vectors for broad applications like retrieval, clustering, and classification.

<img src="https://model-demo.oss-cn-hangzhou.aliyuncs.com/Qwen3-VL-Embedding.png" width="500"/>

In this tutorial we consider how to convert Qwen3-VL Embedding model using Optimum Intel and deploy it with OpenVINO GenAI `EmbeddingPipeline`.

#### Table of contents:

- [Prerequisites](#Prerequisites)
- [Select model](#Select-model)
- [Convert model using Optimum Intel](#Convert-model-using-Optimum-Intel)
- [Run OpenVINO model inference with OpenVINO GenAI](#Run-OpenVINO-model-inference-with-OpenVINO-GenAI)


<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3-vl-embedding/qwen3-vl-embedding.ipynb" />

### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

## Prerequisites
[back to top ⬆️](#Table-of-contents:)

In [ ]:
%pip uninstall -q -y optimum optimum-intel optimum-onnx
%pip install -q "optimum-intel[openvino] @ git+https://github.com/huggingface/optimum-intel.git@f48d93fddff8c91e198389c47a6d5974789b67f4" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q "transformers>=4.57,<=5.0" "torch>=2.9" "torchvision" "qwen-vl-utils>=0.0.14" "pillow" --extra-index-url https://download.pytorch.org/whl/cpu
%pip install -q --pre -U openvino openvino-tokenizers openvino-genai --extra-index-url https://storage.openvinotoolkit.org/simple/wheels/nightly

In [ ]:
import requests
from pathlib import Path

if not Path("cmd_helper.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/cmd_helper.py")
    open("cmd_helper.py", "w").write(r.text)

if not Path("notebook_utils.py").exists():
    r = requests.get(url="https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/notebook_utils.py")
    open("notebook_utils.py", "w").write(r.text)

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3-vl-embedding.ipynb")

## Select model
[back to top ⬆️](#Table-of-contents:)

Qwen3-VL Embedding Model list:

| Model Type | Models | Size | Layers | Sequence Length | Embedding Dimension | MRL Support | Instruction Aware |
|---|---|---|---|---|---|---|---|
| Multimodal Embedding | [Qwen3-VL-Embedding-2B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B) | 2B | 28 | 32K | 2048 | Yes | Yes |
| Multimodal Embedding | [Qwen3-VL-Embedding-8B](https://huggingface.co/Qwen/Qwen3-VL-Embedding-8B) | 8B | 36 | 32K | 4096 | Yes | Yes |

In [ ]:
import ipywidgets as widgets

model_ids = ["Qwen/Qwen3-VL-Embedding-2B", "Qwen/Qwen3-VL-Embedding-8B"]

model_selector = widgets.Dropdown(
    options=model_ids,
    default=model_ids[0],
    description="Embedding Model:",
)

model_selector

## Convert model using Optimum Intel
[back to top ⬆️](#Table-of-contents:)

For convenience, we will use OpenVINO integration with HuggingFace Optimum. [Optimum Intel](https://huggingface.co/docs/optimum/intel/index) is the interface between the Transformers and Diffusers libraries and the different tools and libraries provided by Intel to accelerate end-to-end pipelines on Intel architectures.

Among other use cases, Optimum Intel provides a simple interface to optimize your Transformers and Diffusers models, convert them to the OpenVINO Intermediate Representation (IR) format and run inference using OpenVINO Runtime. `optimum-cli` provides command line interface for model conversion and optimization.

General command format:

```bash
optimum-cli export openvino --model <model_id_or_path> --task <task> <output_dir>
```

where task is task to export the model for. Additionally, you can specify weights compression using `--weight-format` argument with one of following options: `fp32`, `fp16`, `int8` and `int4`.

In [ ]:
to_compress = widgets.Checkbox(
    value=False,
    description="Weight compression",
    disabled=False,
)

visible_widgets = [to_compress]

options = widgets.VBox(visible_widgets)

options

The Qwen3-VL-Embedding model can be exported by `image-text-to-text` task with Optimum-intel. This task exports both the language and vision components, which are required for multimodal (text, image and video) embedding with OpenVINO GenAI.

In [ ]:
from pathlib import Path

model_id = model_selector.value

model_base_dir = Path(model_id.split("/")[-1])
additional_args = {"task": "image-text-to-text"}

if to_compress.value:
    model_dir = model_base_dir / "INT8"
    additional_args.update({"weight-format": "int8"})
else:
    model_dir = model_base_dir / "FP16"
    additional_args.update({"weight-format": "fp16"})

In [ ]:
from cmd_helper import optimum_cli

if not model_dir.exists():
    optimum_cli(model_id, model_dir, additional_args=additional_args)

## Run OpenVINO model inference with OpenVINO GenAI
[back to top ⬆️](#Table-of-contents:)

[OpenVINO GenAI](https://github.com/openvinotoolkit/openvino.genai) provides the `EmbeddingPipeline` class for computing embeddings of text, image and video inputs (including mixed modalities) with Qwen3-VL-Embedding models.

Select device from dropdown list for running inference using OpenVINO.

In [ ]:
from notebook_utils import device_widget

device = device_widget(default="CPU", exclude=["NPU"])
device

The Qwen3-VL-Embedding model can be loaded with the `EmbeddingPipeline` class of OpenVINO GenAI.

In [ ]:
import openvino_genai

pipe = openvino_genai.EmbeddingPipeline(str(model_dir), device.value)

In [ ]:
from io import BytesIO

import numpy as np
import openvino as ov
import requests
from PIL import Image

INSTRUCTION = "Represent the user's input."


def load_image(src):
    """Load an image from a URL or local path as an OpenVINO tensor."""
    if isinstance(src, str) and src.startswith("http"):
        image = Image.open(BytesIO(requests.get(src, timeout=30).content))
    else:
        image = Image.open(src)
    return ov.Tensor(np.array(image.convert("RGB")))


def get_embedding(pipe, inp, instruction=INSTRUCTION):
    """Get a normalized embedding for a single input (text, image, or text+image)."""
    text = None
    images = []
    if isinstance(inp, dict):
        text = inp.get("text")
        if "image" in inp:
            images = [load_image(inp["image"])]
    elif isinstance(inp, str):
        text = inp

    kwargs = {"embedding_prompt": instruction}
    if images:
        kwargs["images"] = images

    result = pipe.embed(text, **kwargs) if text is not None else pipe.embed(**kwargs)

    embedding = np.asarray(result.embeddings.data, dtype=np.float32).reshape(1, -1)
    return embedding / np.linalg.norm(embedding, axis=1, keepdims=True)


# Example from https://huggingface.co/Qwen/Qwen3-VL-Embedding-2B#basic-usage-example
queries = [
    {"text": "A woman playing with her dog on a beach at sunset."},
    {"text": "Pet owner training dog outdoors near water."},
    {"text": "Woman surfing on waves during a sunny day."},
    {"text": "City skyline view from a high-rise building at night."},
]

documents = [
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust."
    },
    {"image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"},
    {
        "text": "A woman shares a joyful moment with her golden retriever on a sun-drenched beach at sunset, as the dog offers its paw in a heartwarming display of companionship and trust.",
        "image": "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg",
    },
]

# Inputs mix text, image, and text+image modalities so we compute embeddings one at a time.
all_inputs = queries + documents
embeddings = np.concatenate([get_embedding(pipe, inp) for inp in all_inputs], axis=0)

num_queries = len(queries)
query_embeddings = embeddings[:num_queries]
doc_embeddings = embeddings[num_queries:]

# Compute similarity scores
scores = query_embeddings @ doc_embeddings.T
print("Similarity scores (queries x documents):")
print(scores.tolist())

### Compare with original PyTorch model (optional)
[back to top ⬆️](#Table-of-contents:)

Tick the checkbox below to also load the original PyTorch model and compare its similarity score matrix with the one produced by the OpenVINO GenAI deployment. For each query, both the reference model and the OpenVINO GenAI pipeline should rank the documents in the same order (i.e. select the same most relevant document).

> **Note:** this cell is **disabled by default** because loading the full PyTorch model requires ~8 GB of memory and adds a few minutes of runtime.


In [ ]:
import ipywidgets as widgets

run_pt_compare_widget = widgets.Checkbox(
    value=False,
    description="Run PyTorch FP32 comparison",
    disabled=False,
)

run_pt_compare_widget

In [ ]:
if not run_pt_compare_widget.value:
    print("Skipping PyTorch comparison. Tick the checkbox above and re-run this cell to enable.")
else:
    import inspect

    import torch
    import torch.nn.functional as F
    from qwen_vl_utils import process_vision_info
    from transformers import AutoModel, AutoProcessor

    processor = AutoProcessor.from_pretrained(model_id)

    def last_token_pool(last_hidden_states, attention_mask):
        left_padding = attention_mask[:, -1].sum() == attention_mask.shape[0]
        if left_padding:
            return last_hidden_states[:, -1]
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

    def get_embedding_pt(model, processor, inp, instruction=INSTRUCTION):
        """PyTorch variant: strip processor keys the Qwen3VLModel forward does not accept
        (e.g. ``mm_token_type_ids``). Mirrors :func:`get_embedding`."""
        content = []
        if isinstance(inp, dict):
            if "image" in inp:
                content.append({"type": "image", "image": inp["image"]})
            if "text" in inp:
                content.append({"type": "text", "text": inp["text"]})
        elif isinstance(inp, str):
            content.append({"type": "text", "text": inp})

        conversation = [
            {"role": "system", "content": [{"type": "text", "text": instruction}]},
            {"role": "user", "content": content},
        ]
        prompt = processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=True)
        image_inputs = process_vision_info(conversation)[0]
        if image_inputs:
            inputs = processor(text=[prompt], images=image_inputs, return_tensors="pt")
        else:
            inputs = processor(text=[prompt], return_tensors="pt")

        accepted = set(inspect.signature(model.forward).parameters)
        call_kwargs = {k: v for k, v in inputs.items() if k in accepted}

        with torch.no_grad():
            outputs = model(**call_kwargs)
        embedding = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
        return F.normalize(embedding, p=2, dim=1)

    # AutoModel loads the Qwen3VLModel backbone that produces the reference last_hidden_state embeddings.
    pt_model = AutoModel.from_pretrained(model_id, dtype=torch.float32).eval()

    pt_embeddings = []
    for inp in all_inputs:
        pt_embeddings.append(get_embedding_pt(pt_model, processor, inp))
    pt_embeddings = torch.cat(pt_embeddings, dim=0)
    pt_scores = (pt_embeddings[:num_queries] @ pt_embeddings[num_queries:].T).numpy()

    print("PyTorch reference similarity scores:")
    print([[round(v, 4) for v in row] for row in pt_scores.tolist()])
    print("OpenVINO GenAI similarity scores:")
    print([[round(v, 4) for v in row] for row in scores.tolist()])

    pt_ranking = pt_scores.argmax(axis=1)
    ov_ranking = scores.argmax(axis=1)
    print("Most relevant document per query (PyTorch):", pt_ranking.tolist())
    print("Most relevant document per query (OpenVINO GenAI):", ov_ranking.tolist())
    print("Retrieval rankings match:", bool((pt_ranking == ov_ranking).all()))

    del pt_model
    import gc

    gc.collect()